# Gold dimensional model and summaries
Maintain dimensions, preserve sales occurrences and update business totals.

## Imports and project setup
Resolve the project directory and load dependencies.

In [ ]:
import sys
from pathlib import Path

project_root = Path(dbutils.widgets.getAll().get(
    "project_root", "/Workspace/Users/allanjs3@yahoo.com.br/beverage_sales"
))
if not project_root.is_absolute() or not (project_root / "configs/gold").is_dir():
    raise ValueError("Project Root must contain configs/gold.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Standard IPython support for interactive edits to driver-side modules.
get_ipython().run_line_magic("reload_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

from utils.execution import finish_execution, start_execution, to_json
from utils.widgets import build_table_dropdown, resolve_table_selection
from transformations.gold.configuration import load_gold_config
from transformations.gold.processing import process_sources
from utils.presentation import render_gold_summary


## Widgets
Select one, several or all configured tables.

In [ ]:
dbutils.widgets.text("project_root", "/Workspace/Users/allanjs3@yahoo.com.br/beverage_sales", "Project Root")
previous_widgets = dbutils.widgets.getAll()
table_dropdown = build_table_dropdown(
    project_root / "configs/gold",
    previous_value=previous_widgets.get("table_name"),
    reset=globals().get("_selector_project_root", str(project_root)) != str(project_root),
)
selector_signature = ("multiselect", str(project_root), tuple(table_dropdown["choices"]))
selector_changed = (
    globals().get("_selector_project_root", str(project_root)) != str(project_root)
    or globals().get("_selector_signature", selector_signature) != selector_signature
)
_selector_project_root = str(project_root)
_selector_signature = selector_signature
# Widget removal and creation must occur in separate Databricks cells.
if selector_changed and "table_name" in previous_widgets:
    dbutils.widgets.remove("table_name")
    table_dropdown["default_value"] = "Select a table"

In [ ]:
dbutils.widgets.multiselect("table_name", table_dropdown["default_value"].split(",")[0], table_dropdown["choices"], "Gold Tables")


## Table configuration
Load table contracts and order the selected dependencies.

In [ ]:
if str(Path(dbutils.widgets.get("project_root"))) != str(project_root):
    raise ValueError("Refresh the parameter cells after changing Project Root.")
selection = resolve_table_selection(dbutils.widgets.get("table_name"), table_dropdown["choices"], "Gold Tables")
config = load_gold_config(project_root / "configs/gold", selection)
print("Selected tables: " + ", ".join(config["selected"]))
execution = start_execution()


## Incremental processing
Process only the selected tables using their independent progress.

In [ ]:
result = process_sources(spark, dbutils, config, execution)


## Execution outcome
Report inserted dimension members, appended sales and written summary groups.

In [ ]:
result = finish_execution(result)
displayHTML(render_gold_summary(result))


## Automation result
Return the structured execution result.

In [ ]:
dbutils.jobs.taskValues.set(key="has_new_data", value=bool(result["has_new_data"]))
dbutils.notebook.exit(to_json(result))
